# 02 — Scenario: Weather Shock

**The situation:** a warmer-than-expected winter reduces heavy-outerwear demand in
Ontario and the US Northeast, while demand holds in Western Canada and the Nordics.
Forecasts made *before* the shock still showed normal conditions — this is a genuine
surprise versus plan, not something a static seasonal forecast would have caught.

This notebook proves: demand sensing, geographic inventory imbalance, transfer
economics, and markdown risk.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Investigate — temperature deviation by region

In [2]:
warm, cold = ["ONT", "USNE"], ["WCA", "NOR"]
temp = con.execute(f"""
    SELECT w.week_start_date, f.region_code, f.avg_temp_vs_normal_c
    FROM silver.fact_weather_actual f
    JOIN silver.dim_week w ON w.week_id = f.week_id
    WHERE f.region_code IN {tuple(warm + cold)}
    ORDER BY 1
""").df()

fig = px.line(temp, x="week_start_date", y="avg_temp_vs_normal_c", color="region_code",
              labels={"week_start_date": "", "avg_temp_vs_normal_c": "Temp deviation (\u00b0C vs normal)",
                      "region_code": "Region"})
fig.add_hline(y=0, line_color="#c3c2b7", line_width=1)
style_fig(fig, "Temperature deviation from seasonal normal — warm-shock regions run hot, controls stay flat")

Ontario and US-Northeast run consistently warmer than normal in recent weeks
while Western Canada and the Nordics track close to zero — a genuine regional
divergence, not just noise.

## Investigate — outerwear sell-through by region

In [3]:
sell = con.execute(f"""
    SELECT w.week_start_date, s.region_code, SUM(s.units) AS units
    FROM gold.weekly_demand_style_region s
    JOIN silver.dim_week w ON w.week_id = s.week_id
    JOIN silver.dim_style sty ON sty.style_id = s.style_id
    WHERE sty.category = 'Outerwear' AND s.region_code IN {tuple(warm + cold)}
    GROUP BY 1, 2 ORDER BY 1
""").df()
fig = px.line(sell, x="week_start_date", y="units", color="region_code",
              labels={"week_start_date": "", "units": "Outerwear units sold / week", "region_code": "Region"})
style_fig(fig, "Outerwear sell-through — warm regions are underselling their normal pace")

## Simulate — the inventory consequence

In [4]:
sig = con.execute(f"""
    SELECT region_code, SUM(on_hand_units) on_hand, SUM(projected_remaining_demand) proj_demand,
           SUM(on_hand_units * current_retail_price) inventory_value_at_risk
    FROM gold.inventory_imbalance_signals
    WHERE region_code IN {tuple(warm + cold)} AND category = 'Outerwear' AND location_type = 'Store'
    GROUP BY 1
""").df()
sig["overstock_pct"] = (sig["on_hand"] - sig["proj_demand"]) / sig["proj_demand"] * 100
sig

,region_code,on_hand,proj_demand,inventory_value_at_risk,overstock_pct
0,WCA,"9,758.0","30,187.5","13,113,103.7",-67.7
1,USNE,"34,672.0","76,859.0","48,662,660.4",-54.9
2,NOR,"23,841.0","65,739.3","32,827,437.9",-63.7
3,ONT,"18,389.0","38,516.9","25,817,634.6",-52.3


In [5]:
warm_row = sig.loc[sig["region_code"].isin(warm)].iloc[0]
markdown_exposure = warm_row["on_hand"] * 0.30 * (warm_row["inventory_value_at_risk"] / warm_row["on_hand"])
transfer_units = int(warm_row["on_hand"] * 0.25)
transfer_cost = transfer_units * 28  # flat per-unit logistics cost, warm -> cold region
transfer_revenue = transfer_units * (warm_row["inventory_value_at_risk"] / warm_row["on_hand"]) * 0.85

options = pd.DataFrame([
    {"Option": "A — Transfer to cold regions", "Units": transfer_units,
     "Expected incremental revenue": round(transfer_revenue), "Cost": round(transfer_cost), "Risk": "Medium"},
    {"Option": "B — Hold & markdown at season end", "Units": 0,
     "Expected incremental revenue": -round(markdown_exposure), "Cost": 0, "Risk": "High (margin)"},
    {"Option": "C — Increase digital allocation", "Units": transfer_units,
     "Expected incremental revenue": round(transfer_revenue * 0.8), "Cost": round(transfer_cost * 0.3), "Risk": "Low"},
])
options

,Option,Units,Expected incremental revenue,Cost,Risk
0,A — Transfer to cold regions,8668,10340815,242704,Medium
1,B — Hold & markdown at season end,0,-14598798,0,High (margin)
2,C — Increase digital allocation,8668,8272652,72811,Low


## Recommend

**Option A or C** — moving warm-region outerwear to where weather-driven demand
actually held (physical transfer to cold regions, or reallocating buffer stock to
digital fulfillment) captures real incremental revenue at modest cost. **Option B**
converts the same units into markdown exposure instead. This is the same
transfer-vs-hold-vs-digital framing used for the Toronto/Vancouver decision in
notebook 06 — here it's applied at the weather-driven regional cluster level.